# Section 1: Import libraries and load CSV dataset

In [1]:
# Import all libraries
import matplotlib as plt
import pandas as pd
import numpy as num
import seaborn as sns
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Download data set
ticker = "AMD"
df = yf.download(
    ticker,
    start = "2018-06-01",
    end = "2026-06-01"
)

df.to_csv("amd_stock_price.csv", index=True)
print("CSV file saved successfully")

[*********************100%***********************]  1 of 1 completed

CSV file saved successfully


# Section 2: Data Cleaning & Preprocessing

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_PATH = "amd_stock_price.csv"  # change this to match where the file sits on your machine

df = pd.read_csv(RAW_PATH, skiprows=[1, 2], header=0)
df.columns = ["Date", "Close", "High", "Low", "Open", "Volume"]
df.head()


print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.info()
df.isnull().sum().sort_values(ascending=False) #find the empty rows

#check for duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)

clean_df = df.copy()
clean_df.head(2)

#clean column names and convert types
clean_df.columns = clean_df.columns.str.strip().str.lower()
clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")
clean_df = clean_df.sort_values("date").reset_index(drop=True)
clean_df.dtypes

#deal with missing values
missing = clean_df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

numeric_cols = ["close", "high", "low", "open", "volume"]

for col in numeric_cols:
    if clean_df[col].isnull().any():
        median_val = clean_df[col].median()
        clean_df[col] = clean_df[col].fillna(median_val)
        print(f"Filled {col} missing values with median: {median_val:.2f}")

print("\nRemaining missing values:")
print(clean_df[numeric_cols].isnull().sum())

#handle outliers

clean_df["return"] = clean_df["close"].pct_change()

q_low, q_high = clean_df["return"].quantile([0.001, 0.999])
n_extreme = ((clean_df["return"] < q_low) | (clean_df["return"] > q_high)).sum()
print(f"Extreme daily return outliers found: {n_extreme}")

clean_df["return"] = clean_df["return"].clip(q_low, q_high)

#create new columns for cleaner
for lag in [1, 2, 3, 5]:
    clean_df[f"close_lag{lag}"] = clean_df["close"].shift(lag)

clean_df["sma_5"] = clean_df["close"].shift(1).rolling(window=5).mean()
clean_df["sma_10"] = clean_df["close"].shift(1).rolling(window=10).mean()
clean_df["volatility_5"] = clean_df["return"].shift(1).rolling(window=5).std()
clean_df["volume_lag1"] = clean_df["volume"].shift(1)
clean_df["volume_sma_5"] = clean_df["volume"].shift(1).rolling(window=5).mean()
clean_df["hl_range_lag1"] = clean_df["high"].shift(1) - clean_df["low"].shift(1)

# Target: next trading day's closing price
clean_df["target"] = clean_df["close"].shift(-1)

clean_df.tail()

#remove structural missing values
before = len(clean_df)
clean_df = clean_df.dropna().reset_index(drop=True)
after = len(clean_df)
print(f"Dropped {before - after} rows with structural NaNs (feature warm-up period + final row with no target)")
print(f"Remaining rows: {after}")

clean_df.to_csv("amd_cleaned.csv", index=False)
print("Saved cleaned file: amd_cleaned.csv")

Shape: (2009, 6)

Columns:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2009 non-null   object 
 1   Close   2009 non-null   float64
 2   High    2009 non-null   float64
 3   Low     2009 non-null   float64
 4   Open    2009 non-null   float64
 5   Volume  2009 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 94.3+ KB
Number of duplicated rows: 0

Remaining missing values:
close     0
high      0
low       0
open      0
volume    0
dtype: int64
Extreme daily return outliers found: 6
Dropped 11 rows with structural NaNs (feature warm-up period + final row with no target)
Remaining rows: 1998
Saved cleaned file: amd_cleaned.csv


# Section 3: Feature Engineering

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

df = pd.read_csv("amd_cleaned.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df_feat = df.copy()

# RSI (Relative Strength Index) - measures market momentum
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df_feat["rsi_14"] = compute_rsi(df_feat["close"].shift(1), window=14)

# EMA - tracks price trend
df_feat["ema_12"] = df_feat["close"].shift(1).ewm(span=12, adjust=False).mean()
df_feat["ema_26"] = df_feat["close"].shift(1).ewm(span=26, adjust=False).mean()

# MACD - trend momentum
df_feat["macd"] = df_feat["ema_12"] - df_feat["ema_26"]
df_feat["macd_signal"] = df_feat["macd"].ewm(span=9, adjust=False).mean()
df_feat["macd_hist"] = df_feat["macd"] - df_feat["macd_signal"]

# Bollinger Bands - check volatility of price, in relation with recent changes
bb_window = 20
bb_mid = df_feat["close"].shift(1).rolling(bb_window).mean()
bb_std = df_feat["close"].shift(1).rolling(bb_window).std()
df_feat["bb_upper"] = bb_mid + 2 * bb_std
df_feat["bb_lower"] = bb_mid - 2 * bb_std
df_feat["bb_width"] = df_feat["bb_upper"] - df_feat["bb_lower"]
df_feat["bb_pct_b"] = (df_feat["close"].shift(1) - df_feat["bb_lower"]) / df_feat["bb_width"]

# ATR (Average True Range) - measures volatility of stock 
prev_close = df_feat["close"].shift(1)
high_low = df_feat["high"].shift(1) - df_feat["low"].shift(1)
high_prevclose = (df_feat["high"].shift(1) - prev_close).abs()
low_prevclose = (df_feat["low"].shift(1) - prev_close).abs()
true_range = pd.concat([high_low, high_prevclose, low_prevclose], axis=1).max(axis=1)
df_feat["atr_14"] = true_range.rolling(14).mean()

# HLC / OC Ratios - measures movement of stock throughout one day
df_feat["hlc_ratio"] = (df_feat["high"].shift(1) - df_feat["low"].shift(1)) / df_feat["close"].shift(1)
df_feat["oc_ratio"] = (df_feat["open"].shift(1) - df_feat["close"].shift(1)) / df_feat["close"].shift(1)

# ROC / Momentum - checks rate price changes
df_feat["roc_5"] = df_feat["close"].shift(1).pct_change(5)
df_feat["roc_10"] = df_feat["close"].shift(1).pct_change(10)
df_feat["momentum_5"] = df_feat["close"].shift(1) - df_feat["close"].shift(6)

# Ratio features - price/volume relative to their recent averages
df_feat["close_to_sma5"] = df_feat["close"].shift(1) / df_feat["sma_5"] - 1
df_feat["close_to_sma10"] = df_feat["close"].shift(1) / df_feat["sma_10"] - 1
df_feat["volume_to_volsma5"] = df_feat["volume"].shift(1) / df_feat["volume_sma_5"] - 1

# Return distribution
df_feat["return_skew_10"] = df_feat["return"].shift(1).rolling(10).skew()
df_feat["return_kurt_10"] = df_feat["return"].shift(1).rolling(10).kurt()

# Rolling min/max - looks at recent suport/resistance levels
df_feat["rolling_max_10"] = df_feat["high"].shift(1).rolling(10).max()
df_feat["rolling_min_10"] = df_feat["low"].shift(1).rolling(10).min()

# Calendar features
df_feat["day_of_week"] = df_feat["date"].dt.dayofweek
df_feat["month"] = df_feat["date"].dt.month
df_feat["is_monday"] = (df_feat["day_of_week"] == 0).astype(int)
df_feat["is_month_end"] = df_feat["date"].dt.is_month_end.astype(int)

# Drop NaNs
before = len(df_feat)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Dropped {before - len(df_feat)} rows due to feature warm-up periods")
print(f"Remaining rows: {len(df_feat)}")

# Correlation check
corr_with_target = df_feat.corr(numeric_only=True)["target"].sort_values(ascending=False)
print("\nTop correlated features with target:")
print(corr_with_target.head(15))

df_feat.to_csv("amd_features.csv", index=False)
print("Saved amd_features.csv")

# TEST / TRAIN / SPLIT
train_df, test_df = train_test_split(df_feat, test_size=0.2, shuffle=False)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\nTrain: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df)} rows)")
print(f"Test:  {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df)} rows)")

# ENCODING
categorical_cols = ["day_of_week", "month"]

train_df = pd.get_dummies(train_df, columns=categorical_cols, prefix=categorical_cols)
test_df = pd.get_dummies(test_df, columns=categorical_cols, prefix=categorical_cols)

train_cols = train_df.columns
train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)
test_df = test_df[train_cols]

# FEATURE SCALING
no_scale_cols = ["date", "target", "is_monday", "is_month_end"]
no_scale_cols += [c for c in train_df.columns if c.startswith("day_of_week_") or c.startswith("month_")]
feature_cols = [c for c in train_df.columns if c not in no_scale_cols]

print(f"\nScaling {len(feature_cols)} numeric features")

scaler = StandardScaler()
train_scaled = train_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

# Save df
train_scaled.to_csv("amd_train_scaled.csv", index=False)
test_scaled.to_csv("amd_test_scaled.csv", index=False)
print("Saved train file: amd_train_scaled.csv")
print("Saved test file: amd_test_scaled.csv")



Dropped 20 rows due to feature warm-up periods
Remaining rows: 1978

Top correlated features with target:
target            1.000000
close             0.997456
low               0.996962
high              0.996944
open              0.996142
close_lag1        0.995085
close_lag2        0.992509
sma_5             0.991786
close_lag3        0.989619
rolling_max_10    0.989147
sma_10            0.987419
ema_12            0.987360
close_lag5        0.984258
bb_upper          0.982643
rolling_min_10    0.980648
Name: target, dtype: float64
Saved amd_features.csv

Train: 2018-07-16 00:00:00 to 2024-10-25 00:00:00 (1582 rows)
Test:  2024-10-28 00:00:00 to 2026-05-28 00:00:00 (396 rows)

Scaling 39 numeric features
Saved train file: amd_train_scaled.csv
Saved test file: amd_test_scaled.csv


# Section 4: Model Training

In [11]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

train_df = pd.read_csv("amd_train_scaled.csv", parse_dates=["date"])
test_df  = pd.read_csv("amd_test_scaled.csv", parse_dates=["date"])

feature_cols = [c for c in train_df.columns if c not in ("date", "target")]

X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test   = test_df[feature_cols], test_df["target"]

tscv = TimeSeriesSplit(n_splits=5)

# Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_mse = mean_squared_error(y_test, lr_pred)
print(f"Linear Regression Test MSE: {lr_mse:.4f}")

# Random Forest Regressor
# hyperparameter for grid
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
print("Random Forest Best params:", rf_grid.best_params_)

rf_pred = rf_best.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
print(f"Random Forest Test MSE: {rf_mse:.4f}")


Linear Regression Test MSE: 85.9265
Random Forest Best params: {'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 100}
Random Forest Test MSE: 3681.7932


# Section 5: Evaluation